<a href="https://colab.research.google.com/github/MoulendraBalaji/Flyrank_ML_Works/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ML-05 — Feature Vector and Leakage/Privacy Check

My lane: **Lane 2 — Refresh / Content Opportunity Scoring**, running on the warehouse release.
This notebook writes down the **feature vector** the lane's model is allowed to learn from, then
**attacks it**: label-derived columns, future windows, and product flags — each one added on
purpose, measured, then thrown out. What survives is what the model may see.

> Skill router: loaded `hunting-leakage-and-validating` + `flyrank/flyrank-data` (per `skills/README.md`).
> Lane continuity: `w03_data_contract.ipynb` fixed the grain (one content page), the windows
> (features = March 2026, label = April 2026) and the label (`will_decline`). This notebook
> keeps that contract and audits its features.

**The rules I follow (from `skills/hunting-leakage-and-validating`):** every feature must be
knowable at the decision moment (**2026-04-01**, when an editor opens the queue); the label lives
strictly after it; the base rate is printed next to every score; and the split is grouped by
client, because pages from one client share hidden character.


## 0. Setup — connect to the warehouse

The token comes from the `HF_TOKEN` environment variable (Colab Secret) or a local `.env` file.
It is **never printed and never pasted in a cell**. The connection is verified the way the
data skill demands: `COUNT(*)` + date span, matched against the release numbers.


In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import duckdb
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import pathlib
    for cand in pathlib.Path.cwd().parents:
        p = cand / ".env"
        if p.exists():
            for line in p.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("HF_TOKEN="):
                    HF_TOKEN = line.strip().split("=", 1)[1].strip().strip('\"').strip("'")
            if HF_TOKEN:
                break
assert HF_TOKEN, "No HF_TOKEN found - set it as an env var / Colab Secret, or a .env file with HF_TOKEN=hf_..."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
F3 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"  # feature month
F4 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"  # outcome month
DC = f"read_parquet('{REL}/dim_content.parquet')"

span = con.sql(f"""
    SELECT COUNT(*), MIN(report_date), MAX(report_date),
           COUNT(DISTINCT client_hash_id), COUNT(DISTINCT content_hash_id)
    FROM {F3}
""").fetchone()
print(f"Connected. Feature month = 2026-03 | outcome month = 2026-04")
print(f"  month=2026-03: {span[0]:,} daily rows, {span[3]} clients, {span[4]:,} content pages")
print(f"  date span: {span[1]} -> {span[2]}")


Connected. Feature month = 2026-03 | outcome month = 2026-04
  month=2026-03: 9,841,378 daily rows, 55 clients, 331,437 content pages
  date span: 2026-03-01 -> 2026-03-31


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

One row = **one content page**. Features come only from what is knowable by **2026-04-01**:

- **numeric** (aggregated over March 2026): `log_imp_mar`, `ctr_mar`, `pos_mar`, `days_mar`, `has_clk_mar`
- **static** (from `dim_content`): `age_days` (page age at the decision moment)
- **categorical** (from `dim_content`): `content_type`, `main_intent` — missing filled with `"unknown"`, then one-hot encoded

Engineered transforms and fills, in the build cell below: `log1p` on impressions (heavy tail),
CTR as a ratio, `has_clk_mar` as an explicit zero/absence flag (a blind fill would encode
missingness as a category signal), and `pos_mar` filled with the median only for pages with no
position readings. The label `will_decline` is built **after** the features, in a separate window.


In [3]:
# 1a. March features, one row per content page, base demand >= 100 impressions.
features = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS imp_mar,
           SUM(gsc_clicks)      AS clk_mar,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_mar,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_mar,
           COUNT(*) AS rows_mar,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_rows_mar
    FROM {F3}
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 100
       AND SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) > 0
""").df()

# 1b. Static metadata + categoricals from dim_content.
meta = con.sql(f"SELECT content_hash_id, content_created_date, content_type, main_intent FROM {DC}").df()
features = features.merge(meta, on="content_hash_id", how="left")

# 1c. Engineered features + fills.
features["age_days"]    = (pd.Timestamp("2026-03-31") - pd.to_datetime(features["content_created_date"])).dt.days
features["ctr_mar"]     = features["clk_mar"] / features["imp_mar"]
features["log_imp_mar"] = np.log1p(features["imp_mar"])
features["has_clk_mar"] = (features["clk_mar"] > 0).astype(int)
features["pos_mar"]     = features["pos_mar"].fillna(features["pos_mar"].median())
for cat in ["content_type", "main_intent"]:
    features[cat] = features[cat].fillna("unknown")

print(f"feature frame: {len(features):,} pages x {features.shape[1]} columns (one row per page)")
print(f"categorical coverage:\n{features[['content_type','main_intent']].value_counts().sort_index().to_string()}")


feature frame: 101,441 pages x 15 columns (one row per page)
categorical coverage:
content_type        main_intent  
comparison article  informational      996
                    transactional        1
feedly article      unknown           1066
keyword article     commercial       17817
                    informational    58904
                    navigational       249
                    transactional    21792
                    unknown            616


### 1d. The label — built after the features, never near them

`will_decline` = April 2026 impressions < 80% of March 2026 impressions (the contract's
definition). The April measurement is the **answer**; the columns it is computed from
(`imp_apr`, `apr_mar_ratio`) are kept on the frame only so the leakage hunt can use them —
they are **not** features.


In [4]:
# 1d. Outcome month (April): the label, and the intermediate columns the label is made of.
apr = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS imp_apr,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END)               AS gsc_rows_apr
    FROM {F4}
    GROUP BY content_hash_id
""").df()

frame = features.merge(apr, on="content_hash_id", how="left")
frame["imp_apr"]       = frame["imp_apr"].fillna(0)
frame["gsc_rows_apr"]  = frame["gsc_rows_apr"].fillna(0)
frame["apr_mar_ratio"] = frame["imp_apr"] / frame["imp_mar"]
frame["will_decline"]  = (frame["imp_apr"] < 0.8 * frame["imp_mar"]).astype(int)

print(f"labeled frame: {len(frame):,} pages | label rate = {frame['will_decline'].mean():.1%} "
      f"(n_pos = {frame['will_decline'].sum():,})")
print(f"pages with no GSC rows in April (counted as 0): {(frame['gsc_rows_apr']==0).sum():,} ")


labeled frame: 101,441 pages | label rate = 51.7% (n_pos = 52,492)
pages with no GSC rows in April (counted as 0): 548 


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing / zero handling | Categorical? | Available before 2026-04-01? |
|---|---|---|---|---|
| `log_imp_mar` | log1p of March GSC impressions — page volume/momentum | none (base demand ≥ 100 guaranteed) | no | yes — March closed |
| `ctr_mar` | March clicks / March impressions — click-worthiness | zero when `clk_mar = 0` | no | yes — both March totals final |
| `pos_mar` | March average GSC position (lower = higher) | median-filled when a page has no position readings | no | yes — March positions final |
| `days_mar` | days in March with ≥ 1 impression — presence consistency | min 1 by construction | no | yes — March fully observed |
| `has_clk_mar` | 1 if March clicks > 0 | explicit absence flag, never blind-filled | no | yes — from March clicks |
| `age_days` | page age in days at the decision moment | none (created date present) | no | yes — static metadata |
| `content_type` | page type (keyword / feedly / comparison article) | `"unknown"` | yes | yes — static metadata |
| `main_intent` | target keyword intent | `"unknown"` | yes | yes — static metadata |

Every feature is knowable at the decision moment because its source window (March, or static
content metadata) has fully closed before 1 April. Nothing above touches April.


In [5]:
# Back the notes with numbers: what fills actually happened, and the final vector shape.
n_pos_fill = int(features['pos_mar'].isna().sum())
print(f"pages whose position was median-filled: {n_pos_fill} ({n_pos_fill/len(features):.1%})")
print(f"pages with zero March clicks (has_clk_mar = 0): {(frame['has_clk_mar']==0).sum():,} "
      f"({(frame['has_clk_mar']==0).mean():.1%})")
print(f"pages with unknown main_intent (fillna): {int((features['main_intent']=='unknown').sum()):,}")
print()
print("numeric feature summary:")
print(frame[["log_imp_mar", "ctr_mar", "pos_mar", "days_mar", "age_days", "has_clk_mar"]].describe().round(3).to_string())


pages whose position was median-filled: 0 (0.0%)
pages with zero March clicks (has_clk_mar = 0): 37,779 (37.2%)
pages with unknown main_intent (fillna): 1,682

numeric feature summary:
       log_imp_mar     ctr_mar     pos_mar    days_mar    age_days  has_clk_mar
count   101441.000  101441.000  101441.000  101441.000  101441.000   101441.000
mean         6.819       0.003      14.765      28.500     188.937        0.628
std          1.412       0.004      14.459       5.022     127.259        0.483
min          4.615       0.000       0.102       1.000       1.000        0.000
25%          5.649       0.000       5.121      29.000      71.000        0.000
50%          6.668       0.001       8.959      31.000     187.000        1.000
75%          7.830       0.004      19.698      31.000     265.000        1.000
max         13.333       0.156     113.390      31.000     494.000        1.000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

One harness, four attacks. Every score sits next to its base rate (label = 51.7% positive, so a
model that always says "declining" is 51.7% "right" — AUC is measured against that, not against
perfection). The harness is a plain logistic regression, fixed seed 42, 80/20 stratified split.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score

NUM = ["log_imp_mar", "ctr_mar", "pos_mar", "days_mar", "age_days", "has_clk_mar"]
CAT = ["content_type", "main_intent"]

def build_X(extra_cols=None, extra_cats=None):
    """The feature matrix: numeric base + one-hot categoricals + (optionally) an attack column."""
    X = frame[NUM].copy()
    for c in (extra_cols or []):
        X[c] = frame[c].values
    for c in CAT + (extra_cats or []):
        X = X.join(pd.get_dummies(frame[c], prefix=c, drop_first=True))
    return X

def quick_score(X, y, seed=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_te, p)

y = frame["will_decline"].values
X_base = build_X()
print(f"base rate (label positive): {y.mean():.1%}")
print(f"HONEST  - the allowed feature vector, random split:")
print(f"   ROC AUC = {quick_score(X_base, y):.4f}")


base rate (label positive): 51.7%
HONEST  - the allowed feature vector, random split:


   ROC AUC = 0.6361


### 3.1 Label-derived columns — the confession test

The label is `will_decline = (imp_apr < 0.8 * imp_mar)`. Two columns are part of that
computation: `apr_mar_ratio` (the ratio itself) and `imp_apr` (its numerator). Neither is
knowable on 1 April. Add them **on purpose** and watch what happens.


In [7]:
print("LEAK test 1 - label-derived columns, added one at a time:")
print(f"   + apr_mar_ratio (the label's own ratio)  AUC = {quick_score(build_X(extra_cols=['apr_mar_ratio']), y):.4f}")
print(f"   + imp_apr (April impressions, numerator) AUC = {quick_score(build_X(extra_cols=['imp_apr']), y):.4f}")
print("   -> apr_mar_ratio alone makes the model PERFECT. That column is the answer, printed as a feature.")
print("   -> imp_apr is part of the answer: a jump from 0.64 to 0.75 is a confession too.")


LEAK test 1 - label-derived columns, added one at a time:


   + apr_mar_ratio (the label's own ratio)  AUC = 1.0000


   + imp_apr (April impressions, numerator) AUC = 0.7554
   -> apr_mar_ratio alone makes the model PERFECT. That column is the answer, printed as a feature.
   -> imp_apr is part of the answer: a jump from 0.64 to 0.75 is a confession too.


### 3.2 Future / overlapping windows

A feature measured **inside the label window** is knowable only after the decision moment. The
label window is all of April. So even **April 1-7** — the week an editor is reviewing the queue
— is the future. Probe with the first week's impressions and average position.


In [8]:
F4w1 = f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet') " \
      f"WHERE report_date BETWEEN '2026-04-01' AND '2026-04-07'"
w1 = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available THEN gsc_impressions ELSE 0 END) AS imp_apr_w1,
           AVG(gsc_avg_position) AS pos_apr_w1
    FROM ({F4w1})
    GROUP BY content_hash_id
""").df()
frame = frame.merge(w1, on="content_hash_id", how="left")
frame["imp_apr_w1"] = frame["imp_apr_w1"].fillna(0)
frame["pos_apr_w1"] = frame["pos_apr_w1"].fillna(frame["pos_apr_w1"].median())

print("LEAK test 2 - future window (April 1-7, inside the label month):")
print(f"   + imp_apr_w1 (week-1 April impressions) AUC = {quick_score(build_X(extra_cols=['imp_apr_w1']), y):.4f}")
print(f"   + pos_apr_w1 (week-1 April position)     AUC = {quick_score(build_X(extra_cols=['pos_apr_w1']), y):.4f}")
print("   -> both lift the score; they read the first 7 days of the answer. Excluded.")


LEAK test 2 - future window (April 1-7, inside the label month):


   + imp_apr_w1 (week-1 April impressions) AUC = 0.7109


   + pos_apr_w1 (week-1 April position)     AUC = 0.6491
   -> both lift the score; they read the first 7 days of the answer. Excluded.


### 3.3 Decision-derived features (product flags)

The product already frames pages with tiers. Rebuild that flag (an `impression_tier`-style bucket
from March impressions) and feed it in. A product flag is a **lossy re-encoding of signals the
model already has** — using it means learning the old rule, not the world. The test: if it adds
no new information, the AUC should barely move, which is itself the circular-result finding.


In [9]:
def imp_tier(imp):
    if imp < 300: return "low"
    if imp < 3000: return "moderate"
    if imp < 30000: return "good"
    return "excellent"
frame["imp_tier_mar"] = frame["imp_mar"].map(imp_tier)

base_auc = quick_score(X_base, y)
flag_auc = quick_score(build_X(extra_cats=["imp_tier_mar"]), y)
print("PRODUCT-flag test - the tier bucket as a feature:")
print(f"   without flag: AUC = {base_auc:.4f}")
print(f"   with    flag: AUC = {flag_auc:.4f}")
print(f"   delta = {flag_auc - base_auc:+.4f} -> the flag carries ~zero new information;")
print("   it only re-encodes impressions (already in log_imp_mar). Using it is circular.")


PRODUCT-flag test - the tier bucket as a feature:
   without flag: AUC = 0.6361
   with    flag: AUC = 0.6346
   delta = -0.0014 -> the flag carries ~zero new information;
   it only re-encodes impressions (already in log_imp_mar). Using it is circular.


### 3.4 Honest splits — random vs client-grouped

Random splits let pages from the same client land in both train and test, and the model can
memorize client character. The honest question is: does it work on a **client it never saw**?
`GroupKFold` holds whole clients out. The gap between the two numbers is a measurement of how
much memorization was happening.


In [10]:
def grouped_auc(X, y, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    preds = np.zeros(len(y))
    groups = frame["client_hash_id"].values
    for tr, te in gkf.split(X, y, groups):
        m = LogisticRegression(max_iter=1000).fit(X.iloc[tr], y[tr])
        preds[te] = m.predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(y, preds)

rand_auc  = quick_score(X_base, y, seed=7)
group_auc = grouped_auc(X_base, y)
print(f"base rate                : {y.mean():.1%}")
print(f"RANDOM split AUC (seed 7): {rand_auc:.4f}")
print(f"CLIENT-GROUPED split AUC : {group_auc:.4f}")
print(f"gap = {rand_auc - group_auc:+.4f} -> some client-specific memorization; "
      f"the grouped number is the deployment-honest one.")


base rate                : 51.7%
RANDOM split AUC (seed 7): 0.6293
CLIENT-GROUPED split AUC : 0.5644
gap = +0.0649 -> some client-specific memorization; the grouped number is the deployment-honest one.


### 3.5 The verdict

| Candidate | What it is | Test result | Verdict |
|---|---|---|---|
| `apr_mar_ratio` | the label's own ratio | AUC 0.64 → **1.00** | **exclude** — it IS the answer |
| `imp_apr` | April impressions | AUC 0.64 → 0.75 | **exclude** — part of the answer |
| `imp_apr_w1` | April week-1 impressions | AUC 0.64 → 0.71 | **exclude** — future window |
| `pos_apr_w1` | April week-1 position | AUC 0.64 → 0.65 | **exclude** — future window |
| `imp_tier_mar` | product tier flag | delta ≈ 0.000 | **exclude** — circular, no new info |
| the 8 real features | — | 0.64 random / **0.56 grouped** | **keep** — honest, modest, decision-support |

The honest number is the **grouped** one: **AUC ≈ 0.56** against a base rate of 51.7% — real but
small skill. The lesson is the confession, not the score: one column from the label window turned
a weak model perfect, and any deployment pipeline that accidentally keeps it would ship fiction.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


In [11]:
excluded = pd.DataFrame([
    ("apr_mar_ratio",    "label-derived",  "the ratio the label thresholds - it IS the answer"),
    ("imp_apr",          "label-derived",  "April impressions - the numerator of the label"),
    ("imp_apr_w1",       "future window",  "April 1-7 impressions - knowable only after 2026-04-01"),
    ("pos_apr_w1",       "future window",  "April 1-7 position - knowable only after 2026-04-01"),
    ("imp_tier_mar",     "product flag",   "rebuild of the product's tier - a lossy re-encode of log_imp_mar"),
    ("fact_content_query_90d", "window overlap", "fixed 90-day window overlaps the label period"),
    ("GA4 columns pre-ga4_data_available", "zero-fill", "zeros before GA4 tracking starts are not 'no engagement'"),
    ("provider_used / model_used", "not a model feature", "generation provenance, not a content signal"),
    ("keyword_hash_id / url_hash_id", "privacy", "pseudonymised identifiers - grouping only, never reconstructed"),
    ("raw queries / titles / domains", "privacy", "never printed, never reconstructed - DATA_USE.md"),
], columns=["field", "reason", "why"])
print(excluded.to_string(index=False))
print()
print("Kept (8 features): log_imp_mar, ctr_mar, pos_mar, days_mar, age_days, has_clk_mar,",
      "content_type, main_intent")


                             field              reason                                                              why
                     apr_mar_ratio       label-derived                the ratio the label thresholds - it IS the answer
                           imp_apr       label-derived                   April impressions - the numerator of the label
                        imp_apr_w1       future window           April 1-7 impressions - knowable only after 2026-04-01
                        pos_apr_w1       future window              April 1-7 position - knowable only after 2026-04-01
                      imp_tier_mar        product flag rebuild of the product's tier - a lossy re-encode of log_imp_mar
            fact_content_query_90d      window overlap                    fixed 90-day window overlaps the label period
GA4 columns pre-ga4_data_available           zero-fill         zeros before GA4 tracking starts are not 'no engagement'
        provider_used / model_used not a

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**One line:** the model may learn from 8 March-2026/static features; the label is April 2026
decline; every attack column (label ratio → AUC 1.00, April week-1 → 0.71, product tier → no
gain) is excluded; the deployment-honest number is the client-grouped AUC ≈ 0.56 against a
51.7% base rate.
